# Residual Multimodal Transformer basline

This experiment converts the visual-first residual GRU fusion model into a
compact multimodal Transformer while preserving the same residual-fusion logic,
fixed subject split, ten-seed stability evaluation, and missing-visual
robustness experiments.

The visual Transformer produces the primary prediction. Compact motion and
heart-rate Transformers can only add a bounded, gated residual correction that
is initialized to zero. This retains a visual-only fallback while evaluating
whether modality-scaled Transformer encoders can exploit complementary sensor
information.


In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

from tqdm import tqdm
import json
import time
import os

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
df = pd.read_parquet("Data/dipser_dataset.parquet", engine="pyarrow")
df["age_group"] = pd.cut(df["age"], bins=[13, 20, 22, 26, 44]) # bins=[13, 17, 20, 22, 26, 44]
df.head()

,group,time,time_sec,image_path,metadata,subject,experiment,self_labeling emotion,self_labeling attention,labeler_02 attention,labeler_04 attention,labeler_02 emotion,labeler_01 attention,labeler_03 emotion,labeler_03 attention,labeler_01 emotion,labeler_04 emotion,self_labeling emotionfilled,self_labeling attentionfilled,labeler_02 attentionfilled,labeler_04 attentionfilled,labeler_02 emotionfilled,labeler_01 attentionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,labeler_04 emotionfilled,samsung_rotation_vector value0_mean,samsung_rotation_vector value0_std,samsung_rotation_vector value1_mean,samsung_rotation_vector value1_std,samsung_rotation_vector value2_mean,samsung_rotation_vector value2_std,samsung_rotation_vector value3_mean,samsung_rotation_vector value3_std,samsung_rotation_vector value4_mean,samsung_rotation_vector value4_std,lsm6dso_gyroscope value0_mean,lsm6dso_gyroscope value0_std,lsm6dso_gyroscope value1_mean,lsm6dso_gyroscope value1_std,lsm6dso_gyroscope value2_mean,lsm6dso_gyroscope value2_std,samsung_linear_acceleration_sensor value0_mean,samsung_linear_acceleration_sensor value0_std,samsung_linear_acceleration_sensor value1_mean,samsung_linear_acceleration_sensor value1_std,samsung_linear_acceleration_sensor value2_mean,samsung_linear_acceleration_sensor value2_std,opt3007_light value0_mean,opt3007_light value0_std,heart_rate,heart_rate_std,accel_magnitude_mean,accel_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled,invalid_reason,gender,age,race,subject_experiment_id,attention,subject_id,age_group
0,group01,2026-05-08 10:40:43.047895,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_047895.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_047895.json,subject_01,experiment01,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109524,0.000163,-0.659717,0.000588,0.632875,0.000495,0.390186,0.000218,246.0,0.0,-0.006022,0.006482,-0.003838,0.003665,-0.000864,0.006845,0.052984,0.034713,-0.132399,0.047077,0.017549,0.043508,63.0,0.0,78.0,NaN,0.155556,0.041540,0.011095,0.005498,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
1,group01,2026-05-08 10:40:43.195317,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_195317.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_195317.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109485,0.000109,-0.660015,0.000520,0.632620,0.000426,0.390106,0.000225,246.0,0.0,-0.004508,0.005235,-0.004068,0.003267,-0.001026,0.005944,0.053056,0.036979,-0.124020,0.045047,0.012282,0.043478,63.0,0.0,78.0,NaN,0.148634,0.038779,0.009572,0.004378,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
2,group01,2026-05-08 10:40:43.295405,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_295405.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_295405.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109478,0.000101,-0.660194,0.000514,0.632473,0.000406,0.390043,0.000252,246.0,0.0,-0.004581,0.005193,-0.003897,0.003153,-0.000525,0.005644,0.054420,0.036670,-0.129478,0.044805,0.015730,0.040968,63.0,0.0,78.0,NaN,0.153129,0.038995,0.009384,0.004079,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
3,group01,2026-05-08 10:40:43.395835,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_395835.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_395835.json,subject_01,experime

In [3]:
df.shape

(889685, 69)

In [4]:
SEED = 42
SEQUENCE_LENGTH = 10
MAX_MISSING_VISUAL_FRAMES = SEQUENCE_LENGTH -1  # remove a sequence only when all the images are unavailable

# Label-imbalance intervention toggles.
USE_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False
WEIGHT_ALPHA = 0.5       # 0=no weighting, 1=full inverse-frequency weighting
STRATIFY_COLUMN = 'age_group'
BATCH_SIZE = 32
NUM_WORKERS = 8

# ATTENTION_BINS = [2, 2.5, 3, 3.5, 4, 4.75]
ATTENTION_BINS = [2, 2.5, 3, 3.5, 4.75]

In [5]:
df[['attention', 'image_path', 'age', 'gender']].isna().sum()

attention     0
image_path    0
age           0
gender        0
dtype: int64

In [6]:
sensor_cols = [
    col for col in df.columns
    if (
       # "sensor" in col.lower()
        "acceleration" in col.lower()
        or "gyro" in col.lower()
      #  or "rotation" in col.lower()
        or "heart" in col.lower()
      #  or "light" in col.lower()
        or "accel" in col.lower()
    )
    and "std" not in col.lower()
]

# Heart rate is physiologically different from motion/device sensors, so Fusion
# follows Temporal Sensor and gives it a separate projection stream.
hr_cols = ["heart_rate"] if "heart_rate" in sensor_cols else []
motion_cols = [col for col in sensor_cols if col not in hr_cols]
hr_indices = [sensor_cols.index(col) for col in hr_cols]
motion_indices = [sensor_cols.index(col) for col in motion_cols]

len(sensor_cols), len(motion_cols), len(hr_cols), sensor_cols[:5]

(9,
 8,
 1,
 ['lsm6dso_gyroscope value0_mean',
  'lsm6dso_gyroscope value1_mean',
  'lsm6dso_gyroscope value2_mean',
  'samsung_linear_acceleration_sensor value0_mean',
  'samsung_linear_acceleration_sensor value1_mean'])

In [7]:
# Treat physiologically impossible HR values as missing before missing flags and scaling.
# The smartwatch can emit 0, which should not be interpreted as a real heart rate.
df.loc[df["heart_rate"] < 30, "heart_rate"] = np.nan

df[sensor_cols] = df[sensor_cols].astype(np.float32)

In [8]:
# keep only 1 frame per second (the first)
df_sec = (df.sort_values(["subject_experiment_id", "time_sec"])
      .groupby(["subject_experiment_id", "time_sec"])
      .first()
      .reset_index())
df_sec.shape

(111754, 69)

In [10]:
reconstructed_sequences = []

for sequence_id, sequence_df in df_sec.groupby("subject_experiment_id"):
    sequence_df = sequence_df.sort_values("time_sec")

    complete_seconds = pd.DataFrame({
        "time_sec": np.arange(
            sequence_df["time_sec"].min(),
            sequence_df["time_sec"].max() + 1
        )
    })

    reconstructed = complete_seconds.merge(
        sequence_df,
        on="time_sec",
        how="left"
    )

    reconstructed["subject_experiment_id"] = sequence_id
    reconstructed_sequences.append(reconstructed)

temporal_frame_dataset = pd.concat(reconstructed_sequences, ignore_index=True)

sequence_metadata = (
    df[["subject_experiment_id", "subject_id", "gender", "age", "age_group"]]
    .drop_duplicates("subject_experiment_id"))

# Restore metadata for reconstructed missing seconds. Sensor/image/target values
# stay missing unless observed, so missingness flags remain meaningful.
temporal_frame_dataset = temporal_frame_dataset.drop(
    columns=["subject_id", "gender", "age", "age_group"],
    errors="ignore").merge(sequence_metadata, on="subject_experiment_id", how="left")

temporal_frame_dataset["visual_missing"] = temporal_frame_dataset["image_path"].isna().astype(np.float32)
temporal_frame_dataset["motion_missing"] = temporal_frame_dataset[motion_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["hr_missing"] = temporal_frame_dataset[hr_cols].isna().all(axis=1).astype(np.float32) if hr_cols else 1.0
temporal_frame_dataset["sensor_missing"] = temporal_frame_dataset[sensor_cols].isna().all(axis=1).astype(np.float32)
temporal_frame_dataset["sensor_partial_nan"] = (
    temporal_frame_dataset[sensor_cols].isna().any(axis=1)
    & ~temporal_frame_dataset[sensor_cols].isna().all(axis=1)
).astype(np.float32)

temporal_frame_dataset[["subject_experiment_id", "time_sec", "visual_missing", "motion_missing", "hr_missing", "sensor_missing", "attention"]].head()

,subject_experiment_id,time_sec,visual_missing,motion_missing,hr_missing,sensor_missing,attention
0,group01_experiment01_subject_01,0,0.0,0.0,0.0,0.0,3.25
1,group01_experiment01_subject_01,1,0.0,0.0,0.0,0.0,3.00
2,group01_experiment01_subject_01,2,0.0,0.0,0.0,0.0,3.00
3,group01_experiment01_subject_01,3,0.0,0.0,0.0,0.0,3.25
4,group01_experiment01_subject_01,4,0.0,0.0,0.0,0.0,3.25


In [11]:
temporal_frame_dataset.shape

(121332, 74)

In [ ]:
feature_index = pd.read_parquet('Data/clip_vitl14_features/clip_vitl14_frame_index.parquet')
feature_store_path =  'Data/clip_vitl14_features/clip_vitl14_frame_features.npy'
VISUAL_FEATURE_DIM = 768

feature_row_by_path = dict(zip(feature_index["image_path"], feature_index["feature_row"]))
FEATURES = feature_store_path.split('/')[-2]

temporal_frame_dataset["feature_row"] = (
    temporal_frame_dataset["image_path"]
    .map(feature_row_by_path)
    .fillna(-1)
    .astype(np.int64))

missing_feature_rows = (
    (temporal_frame_dataset["visual_missing"] == 0)
    & (temporal_frame_dataset["feature_row"] == -1)
).sum()
print("Non-missing image rows without cached features:", missing_feature_rows)

Non-missing image rows without cached features: 0


In [13]:
# Select one fixed age-stratified split using demographic metadata only.
# Model predictions, labels, and test performance are never used to choose it.
SPLIT_SEARCH_TRIALS = 1_000
MIN_MALE_VAL_SUBJECTS = 3
MIN_MALE_TEST_SUBJECTS = 3
MIN_AGE_GROUP_SUBJECTS_VAL = 2
MIN_AGE_GROUP_SUBJECTS_TEST = 2


def demographic_distance(split_df, full_df, column):
    categories = sorted(full_df[column].astype(str).unique())
    full_dist = full_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    split_dist = split_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    return float(np.abs(split_dist - full_dist).sum())


def find_constrained_age_stratified_split(subject_df, trials=SPLIT_SEARCH_TRIALS, seed=SEED):
    candidates = []
    all_age_groups = set(subject_df["age_group"].astype(str))

    for offset in range(trials):
        split_seed = seed + offset
        try:
            train_sub, temp_sub = train_test_split(
                subject_df,
                test_size=0.3,
                stratify=subject_df["age_group"],
                random_state=split_seed,
            )
            val_sub, test_sub = train_test_split(
                temp_sub,
                test_size=0.5,
                stratify=temp_sub["age_group"],
                random_state=split_seed,
            )
        except ValueError:
            continue

        val_males = int((val_sub["gender"].astype(str).str.lower() == "male").sum())
        test_males = int((test_sub["gender"].astype(str).str.lower() == "male").sum())
        val_min_age_count = int(
            val_sub["age_group"].astype(str).value_counts()
            .reindex(all_age_groups, fill_value=0).min()
        )
        test_min_age_count = int(
            test_sub["age_group"].astype(str).value_counts()
            .reindex(all_age_groups, fill_value=0).min()
        )

        representation_penalty = (
            max(0, MIN_MALE_VAL_SUBJECTS - val_males) * 100
            + max(0, MIN_MALE_TEST_SUBJECTS - test_males) * 100
            + max(0, MIN_AGE_GROUP_SUBJECTS_VAL - val_min_age_count) * 100
            + max(0, MIN_AGE_GROUP_SUBJECTS_TEST - test_min_age_count) * 100
        )
        balance_score = sum(
            demographic_distance(split, subject_df, column)
            for split in [train_sub, val_sub, test_sub]
            for column in ["age_group", "gender"]
        )
        candidates.append(
            (
                representation_penalty,
                balance_score,
                split_seed,
                train_sub.copy(),
                val_sub.copy(),
                test_sub.copy(),
            )
        )

    if not candidates:
        raise RuntimeError("Could not construct an age-stratified subject split.")

    best = min(candidates, key=lambda item: (item[0], item[1], item[2]))
    if best[0] > 0:
        raise RuntimeError(
            "No 70-15-15 split satisfied every requested representation constraint."
        )
    return best[3], best[4], best[5], best[2], best[0], best[1]


subject_df = (
    temporal_frame_dataset[["subject_id", "age_group", "gender"]]
    .dropna(subset=["subject_id", "age_group", "gender"])
    .drop_duplicates("subject_id")
    .copy()
)

train_sub, val_sub, test_sub, SPLIT_RANDOM_STATE, split_penalty, split_balance_score = (
    find_constrained_age_stratified_split(subject_df)
)

train_subjects = train_sub["subject_id"]
val_subjects = val_sub["subject_id"]
test_subjects = test_sub["subject_id"]

train_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(train_subjects)].copy()
val_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(val_subjects)].copy()
test_frames = temporal_frame_dataset[temporal_frame_dataset["subject_id"].isin(test_subjects)].copy()

# Train-only sensor normalization, identical to the original notebook.
sensor_means = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].mean()
sensor_stds = train_frames.loc[train_frames["sensor_missing"] == 0, sensor_cols].std()
sensor_stds = sensor_stds.replace(0, np.nan).fillna(1.0)
sensor_means = sensor_means.fillna(0.0)


def apply_sensor_scaling(frame_df):
    frame_df = frame_df.copy()
    scaled = ((frame_df[sensor_cols] - sensor_means) / sensor_stds).astype(np.float32)
    scaled = scaled.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scaled.loc[frame_df["sensor_missing"] == 1, :] = 0.0
    frame_df.loc[:, sensor_cols] = scaled.to_numpy(dtype=np.float32)
    return frame_df


train_frames = apply_sensor_scaling(train_frames)
val_frames = apply_sensor_scaling(val_frames)
test_frames = apply_sensor_scaling(test_frames)

MOTION_INPUT_DIM = len(motion_cols) + 1
HR_INPUT_DIM = len(hr_cols) + 1
SENSOR_INPUT_DIM = len(sensor_cols) + 1


def split_demographic_table(split_df, split_name):
    rows = []
    for attribute in ["age_group", "gender"]:
        for group, count in split_df[attribute].astype(str).value_counts().sort_index().items():
            rows.append({
                "split": split_name,
                "attribute": attribute,
                "group": group,
                "subjects": int(count),
                "proportion": float(count / len(split_df)),
            })
    return pd.DataFrame(rows)


split_demographics = pd.concat(
    [
        split_demographic_table(train_sub, "train"),
        split_demographic_table(val_sub, "validation"),
        split_demographic_table(test_sub, "test"),
    ],
    ignore_index=True,
)

print(f"Selected demographic-only split random state: {SPLIT_RANDOM_STATE}")
print(f"Constraint penalty: {split_penalty}; demographic balance score: {split_balance_score:.4f}")
display(split_demographics)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_frames), len(val_frames), len(test_frames)],
    "subjects": [train_frames.subject_id.nunique(), val_frames.subject_id.nunique(), test_frames.subject_id.nunique()],
    "target_mean": [train_frames.attention.mean(), val_frames.attention.mean(), test_frames.attention.mean()],
    "visual_missing_rate": [train_frames.visual_missing.mean(), val_frames.visual_missing.mean(), test_frames.visual_missing.mean()],
    "motion_missing_rate": [train_frames.motion_missing.mean(), val_frames.motion_missing.mean(), test_frames.motion_missing.mean()],
    "hr_missing_rate": [train_frames.hr_missing.mean(), val_frames.hr_missing.mean(), test_frames.hr_missing.mean()],
    "sensor_missing_rate": [train_frames.sensor_missing.mean(), val_frames.sensor_missing.mean(), test_frames.sensor_missing.mean()],
})


Selected demographic-only split random state: 50
Constraint penalty: 0; demographic balance score: 0.2392


,split,attribute,group,subjects,proportion
0,train,age_group,"(13, 20]",11,0.333333
1,train,age_group,"(20, 22]",9,0.272727
2,train,age_group,"(22, 26]",9,0.272727
3,train,age_group,"(26, 44]",4,0.121212
4,train,gender,female,23,0.696970
5,train,gender,male,10,0.303030
6,validation,age_group,"(13, 20]",4,0.333333
7,validation,age_group,"(20, 22]",3,0.250000
8,validation,age_group,"(22, 26]",3,0.250000
9,validation,age_group,"(26, 44]",2,0.166667


,split,rows,subjects,target_mean,visual_missing_rate,motion_missing_rate,hr_missing_rate,sensor_missing_rate
0,train,69094,33,2.942945,0.089082,0.089082,0.093062,0.089082
1,val,26589,12,2.995684,0.071947,0.071947,0.089812,0.071947
2,test,25649,12,2.938579,0.069281,0.069281,0.103786,0.069281


In [14]:
def create_temporal_sequences(frame_df, sequence_length=SEQUENCE_LENGTH):
    sequences = []

    for sequence_id, sequence_df in frame_df.groupby("subject_experiment_id"):
        sequence_df = sequence_df.sort_values("time_sec").reset_index(drop=True)

        if len(sequence_df) < sequence_length:
            continue

        for i in range(sequence_length - 1, len(sequence_df)):
            target = sequence_df.iloc[i]["attention"]

            if pd.isna(target):
                continue

            history = sequence_df.iloc[i - sequence_length + 1:i + 1]
            visual_missing_flags = history["visual_missing"].astype(np.float32).values

            if visual_missing_flags.sum() > MAX_MISSING_VISUAL_FRAMES:
                continue

            motion_missing_flags = history["motion_missing"].astype(np.float32).values
            hr_missing_flags = history["hr_missing"].astype(np.float32).values
            sensor_missing_flags = history["sensor_missing"].astype(np.float32).values

            sequence_record = {
                "subject_experiment_id": sequence_id,
                "subject_id": sequence_df.iloc[i]["subject_id"],
                "gender": sequence_df.iloc[i]["gender"],
                "age": sequence_df.iloc[i]["age"],
                "age_group": sequence_df.iloc[i]["age_group"],
                "time_sec": int(sequence_df.iloc[i]["time_sec"]),
                "feature_rows": history["feature_row"].tolist(),
                "visual_missing_flags": visual_missing_flags.tolist(),
                "motion_missing_flags": motion_missing_flags.tolist(),
                "hr_missing_flags": hr_missing_flags.tolist(),
                "sensor_missing_flags": sensor_missing_flags.tolist(),
                "target": float(target),
            }

            # Store every scaled sensor as its own temporal column. Each value is
            # a length-SEQUENCE_LENGTH list, e.g. train_df["heart_rate"].iloc[0].
            for sensor_col in sensor_cols:
                sequence_record[sensor_col] = (
                    history[sensor_col]
                    .astype(np.float32)
                    .to_numpy(dtype=np.float32)
                    .tolist()
                )

            sequences.append(sequence_record)

    return pd.DataFrame(sequences)

In [15]:
train_df = create_temporal_sequences(train_frames)
val_df = create_temporal_sequences(val_frames)
test_df = create_temporal_sequences(test_frames)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
})

,split,sequences,subjects,target_mean
0,train,60983,33,2.946022
1,val,23908,12,2.999916
2,test,23142,12,2.940087


### Train Test Split

In [16]:
class MultimodalFusionDataset(Dataset):

    def __init__(self, sequence_df, feature_store_path):
        self.df = sequence_df.reset_index(drop=True)
        self.feature_store_path = feature_store_path
        self.feature_store = None

    def __len__(self):
        return len(self.df)

    def _features(self):
        if self.feature_store is None:
            self.feature_store = np.load(self.feature_store_path, mmap_mode="r")
        return self.feature_store

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64)
        visual_features = np.zeros((len(feature_rows), VISUAL_FEATURE_DIM), dtype=np.float32)

        valid = feature_rows >= 0
        visual_features[valid] = self._features()[feature_rows[valid]]

        # Reconstruct the T x sensor_dim tensor from separate temporal sensor
        # columns instead of reading one packed sensor_values column.
        sensors = np.stack(
            [
                np.asarray(row[sensor_col], dtype=np.float32)
                for sensor_col in sensor_cols
            ],
            axis=1
        )
        sensors = torch.tensor(sensors, dtype=torch.float32)
        sensors = torch.nan_to_num(sensors, nan=0.0, posinf=0.0, neginf=0.0)

        motion = sensors[:, motion_indices]
        motion_missing = torch.tensor(row["motion_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        motion = torch.cat([motion, motion_missing], dim=-1)

        if hr_indices:
            heart_rate = sensors[:, hr_indices]
        else:
            heart_rate = torch.zeros((sensors.shape[0], 0), dtype=torch.float32)
        hr_missing = torch.tensor(row["hr_missing_flags"], dtype=torch.float32).unsqueeze(-1)
        heart_rate = torch.cat([heart_rate, hr_missing], dim=-1)

        visual_features = torch.tensor(visual_features, dtype=torch.float32)
        visual_missing_flags = torch.tensor(row["visual_missing_flags"], dtype=torch.float32)
        target = torch.tensor(row["target"], dtype=torch.float32)
        sample_weight = torch.tensor(row["sample_weight"], dtype=torch.float32)

        return visual_features, motion, heart_rate, visual_missing_flags, target, sample_weight, idx

In [17]:
# Quantify target imbalance and create inverse-frequency train weights.
# Weights are learned from train only; val/test weights are diagnostic only.
def add_attention_bin_weights(alpha, train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_bins = pd.cut(train_df["target"], bins=ATTENTION_BINS, include_lowest=True)
    bin_counts = train_bins.value_counts().sort_index()
    nonzero_counts = bin_counts[bin_counts > 0]

    bin_weights = (len(train_df) / (len(nonzero_counts) * nonzero_counts)) ** alpha

    for split_df in [train_df, val_df, test_df]:
        split_bins = pd.cut(split_df["target"], bins=ATTENTION_BINS, include_lowest=True)
        split_df["attention_bin"] = split_bins.astype(str)
        split_df["sample_weight"] = split_bins.map(bin_weights).astype(float).fillna(1.0)

    train_weight_mean = train_df["sample_weight"].mean()

    for split_df in [train_df, val_df, test_df]:
        split_df["sample_weight"] = split_df["sample_weight"] / train_weight_mean

    normalized_bin_weights = bin_weights / train_weight_mean

    imbalance_table = pd.DataFrame({
        "bin": bin_counts.index.astype(str),
        "train_count": bin_counts.values,
        "weight": [
            float(normalized_bin_weights.get(idx, np.nan))
            for idx in bin_counts.index
        ],
    })

    return train_df, val_df, test_df, imbalance_table

train_df, val_df, test_df, imbalance_table = add_attention_bin_weights(WEIGHT_ALPHA, train_df, val_df, test_df)
display(imbalance_table)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
    "mean_sample_weight": [train_df.sample_weight.mean(), val_df.sample_weight.mean(), test_df.sample_weight.mean()],
})

,bin,train_count,weight
0,"(1.999, 2.5]",13031,1.140073
1,"(2.5, 3.0]",30614,0.743809
2,"(3.0, 3.5]",13128,1.135853
3,"(3.5, 4.75]",4210,2.005766


,split,sequences,subjects,target_mean,mean_sample_weight
0,train,60983,33,2.946022,1.000000
1,val,23908,12,2.999916,1.002191
2,test,23142,12,2.940087,1.023525


In [18]:
train_df.shape, val_df.shape, test_df.shape

((60983, 23), (23908, 23), (23142, 23))

In [19]:
len(set(train_df['subject_experiment_id'])), len(set(val_df['subject_experiment_id'])) ,len(set(test_df['subject_experiment_id']))

(232, 89, 86)

In [20]:
train_dataset = MultimodalFusionDataset(train_df, feature_store_path)
val_dataset = MultimodalFusionDataset(val_df, feature_store_path)
test_dataset = MultimodalFusionDataset(test_df, feature_store_path)

if USE_WEIGHTED_SAMPLER:
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
        num_samples=len(train_df),
        replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4)

### Training

In [21]:
class ResidualMultimodalFusionTransformer(nn.Module):

    def __init__(
        self,
        motion_input_dim,
        hr_input_dim,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        visual_dim=128,
        motion_dim=32,
        hr_dim=16,
        dropout=0.3,
        max_sensor_correction=0.5,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.visual_projection = nn.Sequential(
            nn.Linear(visual_feature_dim + 1, visual_dim),
            nn.LayerNorm(visual_dim),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.motion_projection = nn.Sequential(
            nn.Linear(motion_input_dim, motion_dim),
            nn.LayerNorm(motion_dim),
            nn.GELU(),
        )
        self.hr_projection = nn.Sequential(
            nn.Linear(hr_input_dim, hr_dim),
            nn.LayerNorm(hr_dim),
            nn.GELU(),
        )

        # Each modality receives capacity proportional to its input complexity.
        self.visual_position = nn.Parameter(torch.zeros(max_seq_len, visual_dim))
        self.motion_position = nn.Parameter(torch.zeros(max_seq_len, motion_dim))
        self.hr_position = nn.Parameter(torch.zeros(max_seq_len, hr_dim))

        visual_layer = nn.TransformerEncoderLayer(
            d_model=visual_dim,
            nhead=4,
            dim_feedforward=256,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        motion_layer = nn.TransformerEncoderLayer(
            d_model=motion_dim,
            nhead=4,
            dim_feedforward=64,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        hr_layer = nn.TransformerEncoderLayer(
            d_model=hr_dim,
            nhead=2,
            dim_feedforward=32,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.visual_encoder = nn.TransformerEncoder(visual_layer, num_layers=2)
        self.motion_encoder = nn.TransformerEncoder(motion_layer, num_layers=1)
        self.hr_encoder = nn.TransformerEncoder(hr_layer, num_layers=1)
        self.visual_norm = nn.LayerNorm(visual_dim)
        self.motion_norm = nn.LayerNorm(motion_dim)
        self.hr_norm = nn.LayerNorm(hr_dim)

        self.visual_regressor = nn.Sequential(
            nn.Linear(visual_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

        fusion_dim = visual_dim + motion_dim + hr_dim
        self.sensor_dropout = nn.Dropout(0.25)
        self.residual_gate = nn.Sequential(
            nn.Linear(fusion_dim, 32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        self.sensor_residual = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
        self.max_sensor_correction = max_sensor_correction

        # Begin as a visual-only Transformer. Sensor influence must be learned.
        nn.init.zeros_(self.sensor_residual[-1].weight)
        nn.init.zeros_(self.sensor_residual[-1].bias)
        nn.init.constant_(self.residual_gate[-2].bias, -2.0)

    @staticmethod
    def causal_mask(sequence_length, device):
        positions = torch.arange(sequence_length, device=device)
        return positions.unsqueeze(0) > positions.unsqueeze(1)

    def forward(self, visual_features, motion, heart_rate, visual_missing_flags):
        _, sequence_length, _ = visual_features.shape
        mask = self.causal_mask(sequence_length, visual_features.device)

        visual_missing_flags = visual_missing_flags.unsqueeze(-1)
        visual_input = torch.cat([visual_features, visual_missing_flags], dim=-1)
        visual_tokens = (
            self.visual_projection(visual_input)
            + self.visual_position[:sequence_length].unsqueeze(0)
        )
        motion_tokens = (
            self.motion_projection(motion)
            + self.motion_position[:sequence_length].unsqueeze(0)
        )
        hr_tokens = (
            self.hr_projection(heart_rate)
            + self.hr_position[:sequence_length].unsqueeze(0)
        )

        visual_encoded = self.visual_encoder(visual_tokens, mask=mask)
        motion_encoded = self.motion_encoder(motion_tokens, mask=mask)
        hr_encoded = self.hr_encoder(hr_tokens, mask=mask)

        visual_summary = self.visual_norm(visual_encoded[:, -1, :])
        motion_summary = self.motion_norm(motion_encoded[:, -1, :])
        hr_summary = self.hr_norm(hr_encoded[:, -1, :])
        visual_prediction = self.visual_regressor(visual_summary).squeeze(1)

        fusion_context = torch.cat(
            [visual_summary, motion_summary, hr_summary],
            dim=1,
        )
        fusion_context = self.sensor_dropout(fusion_context)
        gate = self.residual_gate(fusion_context).squeeze(1)
        residual = torch.tanh(self.sensor_residual(fusion_context).squeeze(1))
        return visual_prediction + self.max_sensor_correction * gate * residual


### Fixed-split multi-seed residual Transformer stability

This notebook measures whether visual-first residual fusion improves over the
visual-only baseline without inheriting the instability of the original large
Transformer fusion model.

The fixed split, weighted task loss, batch size, preprocessing, and ten training
seeds remain comparable to the baseline stability notebooks. The intentional
changes are the compact residual architecture, shuffled training batches, MSE
objective, RMSE checkpoint selection, and stronger weight decay.


In [22]:
import copy
import random
from importlib import reload
import src.evaluation as ev

ev = reload(ev)


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_model():
    return ResidualMultimodalFusionTransformer(
        motion_input_dim=MOTION_INPUT_DIM,
        hr_input_dim=HR_INPUT_DIM,
        max_seq_len=SEQUENCE_LENGTH,
        visual_feature_dim=VISUAL_FEATURE_DIM,
        visual_dim=128,
        motion_dim=32,
        hr_dim=16,
        dropout=0.3,
        max_sensor_correction=0.5,
    )


def make_train_loader():
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=TRAIN_SHUFFLE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
        prefetch_factor=4,
    )


def weighted_task_loss(per_sample_loss, sample_weights):
    if USE_WEIGHTED_LOSS:
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return per_sample_loss.mean()


def train_one_epoch_baseline(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    preds_all, labels_all = [], []

    for visual_features, motion, heart_rate, visual_missing_flags, labels, sample_weights, _idx in tqdm(loader, desc="Training", leave=False):
        visual_features = visual_features.to(device)
        motion = motion.to(device)
        heart_rate = heart_rate.to(device)
        visual_missing_flags = visual_missing_flags.to(device)
        labels = labels.to(device)
        sample_weights = sample_weights.to(device)

        preds = model(visual_features, motion, heart_rate, visual_missing_flags)
        loss = weighted_task_loss(criterion(preds, labels), sample_weights)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.item())
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader),
        "mae": float(mean_absolute_error(labels_all, preds_all)),
        "rmse": float(np.sqrt(mean_squared_error(labels_all, preds_all))),
    }


def evaluate_loader(model, loader, sequence_df):
    model.eval()
    preds_all, labels_all, indices_all = [], [], []

    with torch.no_grad():
        for visual_features, motion, heart_rate, visual_missing_flags, labels, _sample_weights, idx in tqdm(loader, desc="Evaluating", leave=False):
            preds = model(
                visual_features.to(device),
                motion.to(device),
                heart_rate.to(device),
                visual_missing_flags.to(device),
            )
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())
            indices_all.extend(idx.detach().cpu().numpy())

    frame = sequence_df.iloc[np.asarray(indices_all, dtype=int)][
        ["subject_experiment_id", "subject_id", "time_sec", "attention_bin", "gender", "age", "age_group"]
    ].reset_index(drop=True)
    frame.insert(0, "true", np.asarray(labels_all, dtype=float))
    frame.insert(0, "pred", np.asarray(preds_all, dtype=float))
    frame = ev.add_robustness_metadata(
        frame, sequence_df, visual_missing_col="visual_missing_flags"
    )

    overall = ev.compute_prediction_metrics(frame)
    age_mae, age_worst, age_gap = ev.compute_group_mae(frame, "age_group")
    gender_mae, gender_worst, gender_gap = ev.compute_group_mae(frame, "gender")
    metrics = {
        "mae": overall["mae"],
        "rmse": overall["rmse"],
        "r2": overall["r2"],
        "true_mean": overall["true_mean"],
        "pred_mean": overall["pred_mean"],
        "age_worst_group_mae": age_worst,
        "age_gap": age_gap,
        "gender_worst_group_mae": gender_worst,
        "gender_gap": gender_gap,
        "age_mae_per_group": age_mae.to_dict(),
        "gender_mae_per_group": gender_mae.to_dict(),
    }
    return metrics, frame

In [ ]:
class EarlyStopping:
    def __init__(self, patience, model_path):
        self.patience = patience
        self.model_path = model_path
        self.best_score = float("inf")
        self.best_epoch = None
        self.counter = 0

    def step(self, val_mae, model, epoch):
        if val_mae < self.best_score:
            self.best_score = float(val_mae)
            self.best_epoch = epoch
            self.counter = 0
            torch.save(copy.deepcopy(model.state_dict()), self.model_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


LOSS_TYPE = "mse"
criterion = nn.MSELoss(reduction="none")
EARLY_STOPPING_METRIC = "rmse"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
PATIENCE = 7
TRAIN_SHUFFLE = True

# Ten independent initialization/training seeds on the exact same subject split.
RUN_SEEDS = [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]

RESULTS_DIR = "results/Multimodal Fusion Residual Transformer Stability"
MODEL_DIR = "models/Multimodal Fusion Residual Transformer Stability"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Fixed split random state: {SPLIT_RANDOM_STATE}")
print(f"Training shuffle: {TRAIN_SHUFFLE}")
print(f"Training seeds: {RUN_SEEDS}")

Fixed split random state: 50
Training shuffle: True
Training seeds: [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]


In [55]:
run_records = []
test_predictions = {}

for run_seed in RUN_SEEDS:
    set_global_seed(run_seed)
    run_name = f"residual_transformer_fixed_age_split_seed{run_seed}"
    model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")

    model = make_model().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.3, patience=3
    )
    early_stopping = EarlyStopping(PATIENCE, model_path)
    run_train_loader = make_train_loader()
    history = []

    print(f"\n=== {run_name} ===")
    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch_baseline(model, run_train_loader, optimizer, criterion)
        val_metrics, _ = evaluate_loader(model, val_loader, val_df)
        monitor = val_metrics[EARLY_STOPPING_METRIC]
        scheduler.step(monitor)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "val_r2": val_metrics["r2"],
        })
        print(
            f"Epoch {epoch + 1:02d} | train MAE {train_metrics['mae']:.4f} | "
            f"val MAE {val_metrics['mae']:.4f} | val R2 {val_metrics['r2']:.4f}"
        )
        if early_stopping.step(monitor, model, epoch):
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    test_metrics, test_frame = evaluate_loader(model, test_loader, test_df)
    test_predictions[run_name] = test_frame

    val_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_val_predictions.csv")
    test_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_test_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_prediction_path)
    ev.save_prediction_frame(test_frame, test_prediction_path)

    run_records.append({
        "run_name": run_name,
        "run_seed": run_seed,
        "best_epoch": early_stopping.best_epoch + 1,
        "num_epochs_run": len(history),
        "model_path": model_path,
        "val_prediction_path": val_prediction_path,
        "test_prediction_path": test_prediction_path,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "history": history,
    })

print("Finished all residual Transformer runs.")

/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



=== residual_transformer_fixed_age_split_seed42 ===


Epoch 01 | train MAE 0.3470 | val MAE 0.3449 | val R2 -0.0757


Epoch 02 | train MAE 0.2726 | val MAE 0.3524 | val R2 -0.1519


Epoch 03 | train MAE 0.2529 | val MAE 0.3458 | val R2 -0.1028


...

Epoch 07 | train MAE 0.2183 | val MAE 0.3696 | val R2 -0.3432


Epoch 08 | train MAE 0.2083 | val MAE 0.3594 | val R2 -0.2444


Epoch 09 | train MAE 0.2049 | val MAE 0.3677 | val R2 -0.2989


Epoch 10 | train MAE 0.2033 | val MAE 0.3519 | val R2 -0.2214
Early stopping triggered


Finished all residual Transformer runs.


In [56]:
metric_rows = []
for run in run_records:
    row = {
        "run_name": run["run_name"],
        "run_seed": run["run_seed"],
        "best_epoch": run["best_epoch"],
        "num_epochs_run": run["num_epochs_run"],
    }
    row.update({f"val_{key}": value for key, value in run["val_metrics"].items() if not isinstance(value, dict)})
    row.update({f"test_{key}": value for key, value in run["test_metrics"].items() if not isinstance(value, dict)})
    metric_rows.append(row)

seed_results = pd.DataFrame(metric_rows)

summary_metrics = [
    "test_mae",
    "test_rmse",
    "test_r2",
    "test_age_worst_group_mae",
    "test_age_gap",
    "test_gender_worst_group_mae",
    "test_gender_gap",
]
seed_summary = (
    seed_results[summary_metrics]
    .agg(["mean", "std", "min", "median", "max"])
    .T
    .reset_index(names="metric")
)

display(seed_results.round(4))
display(seed_summary.round(4))


,run_name,run_seed,best_epoch,num_epochs_run,val_mae,val_rmse,val_r2,val_true_mean,val_pred_mean,val_age_worst_group_mae,val_age_gap,val_gender_worst_group_mae,val_gender_gap,test_mae,test_rmse,test_r2,test_true_mean,test_pred_mean,test_age_worst_group_mae,test_age_gap,test_gender_worst_group_mae,test_gender_gap
0,residual_transformer_fixed_age_split_seed42,42,1,8,0.3449,0.4371,-0.0757,2.9999,3.0505,0.3979,0.1174,0.3551,0.0293,0.3572,0.4565,-0.0825,2.9401,3.1095,0.4078,0.0974,0.3677,0.0296
1,residual_transformer_fixed_age_split_seed100,100,1,8,0.3488,0.4433,-0.1067,2.9999,3.0862,0.4191,0.1352,0.3582,0.0271,0.3803,0.4913,-0.2538,2.9401,3.1362,0.4309,0.0903,0.3967,0.0465
2,residual_transformer_fixed_age_split_seed2000,2000,2,9,0.3430,0.4391,-0.0857,2.9999,3.0332,0.4362,0.1433,0.3439,0.0027,0.3316,0.4312,0.0345,2.9401,3.0688,0.3988,0.1260,0.3586,0.0416
3,residual_transformer_fixed_age_split_seed2025,2025,3,10,0.3410,0.4379,-0.0801,2.9999,2.9688,0.4152,0.1348,0.3522,0.0172,0.3438,0.4391,-0.0012,2.9401,3.0156,0.3842,0.0723,0.3474,0.0055
4,residual_transformer_fixed_age_split_seed2026,2026,4,11,0.3338,0.4305,-0.0439,2.9999,2.9417,0.3829,0.0911,0.3342,0.0013,0.3285,0.4151,0.1053,2.9401,2.9903,0.3681,0.0781,0.3344,0.0092
5,residual_transformer_fixed_age_split_seed2027,2027,3,10,0.3479,0.4486,-0.1333,2.9999,2.9429,0.3709,0.0732,0.3589,0.0170,0.3452,0.4397,-0.0041,2.9401,2.9899,0.3852,0.1008,0.3480,0.0043
6,residual_transformer_fixed_age_split_seed2048,2048,2,9,0.3494,0.4467,-0.1237,2.9999,2.9872,0.3961,0.0937,0.3526,0.0049,0.3332,0.4283,0.0472,2.9401,3.0403,0.3858,0.1169,0.3495,0.0253
7,residual_transformer_fixed_age_split_seed4096,4096,1,8,0.3428,0.4406,-0.0934,2.9999,3.0186,0.4204,0.1702,0.3540,0.0321,0.3412,0.4360,0.0129,2.9401,3.0491,0.3743,0.0839,0.3446,0.0052
8,residual_transformer_fixed_age_split_seed7000,7000,2,9,0.3363,0.4292,-0.0376,2.9999,3.0021,0.3882,0.0783,0.3403,0.0116,0.3304,0.4237,0.0677,2.9401,3.0625,0.3695,0.1002,0.3362,0.0089
9,residual_transformer_fixed_age_split_seed8192,8192,3,10,0.3331,0.4388,-0.0844,2.9999,2.9965,0.4051,0.1497,0.3365,0.0097,0.3205,0.4117,0.1199,2.9401,3.0124,0.3628,0.0913,0.3331,0.0195


,metric,mean,std,min,median,max
0,test_mae,0.3412,0.0172,0.3205,0.3372,0.3803
1,test_rmse,0.4372,0.0230,0.4117,0.4336,0.4913
2,test_r2,0.0046,0.1078,-0.2538,0.0237,0.1199
3,test_age_worst_group_mae,0.3867,0.0209,0.3628,0.3847,0.4309
4,test_age_gap,0.0957,0.0165,0.0723,0.0944,0.1260
5,test_gender_worst_group_mae,0.3516,0.0191,0.3331,0.3477,0.3967
6,test_gender_gap,0.0196,0.0157,0.0043,0.0144,0.0465


In [57]:
mean_baseline_predictions = ev.make_mean_baseline_predictions(train_df, test_df)
mean_baseline_metrics = ev.compute_prediction_metrics(mean_baseline_predictions)

# Diagnostic seed ensemble: average predictions from all independently trained
# models. This is not treated as a single-model baseline.
ensemble_frame = next(iter(test_predictions.values())).copy()
ensemble_frame["pred"] = np.mean(
    [frame["pred"].to_numpy(dtype=float) for frame in test_predictions.values()],
    axis=0,
)
ensemble_metrics = ev.compute_prediction_metrics(ensemble_frame)
_, ensemble_age_worst, ensemble_age_gap = ev.compute_group_mae(ensemble_frame, "age_group")
_, ensemble_gender_worst, ensemble_gender_gap = ev.compute_group_mae(ensemble_frame, "gender")
ensemble_metrics.update({
    "age_worst_group_mae": ensemble_age_worst,
    "age_gap": ensemble_age_gap,
    "gender_worst_group_mae": ensemble_gender_worst,
    "gender_gap": ensemble_gender_gap,
})

reference_table = pd.DataFrame([
    {"model": "train-mean constant baseline", **mean_baseline_metrics},
    {"model": "ten-seed prediction ensemble (diagnostic)", **ensemble_metrics},
])

display(reference_table.round(4))


,model,n_samples,mae,rmse,r2,tolerant_accuracy,tolerance,one_off_accuracy,binary_threshold,binary_accuracy,binary_f1,binary_roc_auc,binary_confusion_matrix,true_mean,pred_mean,age_worst_group_mae,age_gap,gender_worst_group_mae,gender_gap
0,train-mean constant baseline,23142,0.3442,0.4388,-0.0002,0.2287,0.15,1.0000,3.0,0.4719,0.0000,0.5000,"[[10921, 0], [12221, 0]]",2.9401,2.9460,NaN,NaN,NaN,NaN
1,ten-seed prediction ensemble (diagnostic),23142,0.3307,0.4264,0.0557,0.3162,0.15,0.9959,3.0,0.5889,0.6015,0.6272,"[[6450, 4471], [5042, 7179]]",2.9401,3.0474,0.3718,0.0839,0.3373,0.0102


In [58]:
BOOTSTRAP_RUNS = 1000
bootstrap_summary, _bootstrap_samples = ev.bootstrap_table(
    test_predictions,
    cluster_col="subject_id",
    n_boot=BOOTSTRAP_RUNS,
    seed=SEED,
)

# Quantify how much each test subject's MAE changes across training seeds.
subject_rows = []
for run_name, frame in test_predictions.items():
    per_subject = (
        frame.assign(abs_error=np.abs(frame["true"].astype(float) - frame["pred"].astype(float)))
        .groupby("subject_id", observed=True)
        .agg(
            samples=("abs_error", "size"),
            mae=("abs_error", "mean"),
            true_mean=("true", "mean"),
            pred_mean=("pred", "mean"),
        )
        .reset_index()
    )
    per_subject.insert(0, "run_name", run_name)
    subject_rows.append(per_subject)

subject_seed_metrics = pd.concat(subject_rows, ignore_index=True)
subject_stability = (
    subject_seed_metrics.groupby("subject_id", observed=True)
    .agg(
        samples=("samples", "first"),
        true_mean=("true_mean", "first"),
        mean_mae=("mae", "mean"),
        sd_mae=("mae", "std"),
        min_mae=("mae", "min"),
        max_mae=("mae", "max"),
    )
    .reset_index()
    .sort_values("mean_mae", ascending=False)
)

display(bootstrap_summary.round(4))
display(subject_stability.round(4))


,model,metric,mean,ci_low,ci_high
0,residual_transformer_fixed_age_split_seed42,mae,0.3592,0.3041,0.4158
1,residual_transformer_fixed_age_split_seed42,rmse,0.4577,0.3944,0.5171
2,residual_transformer_fixed_age_split_seed42,r2,-0.1343,-0.6275,0.1669
3,residual_transformer_fixed_age_split_seed42,gender_gap,0.0607,0.0013,0.1684
4,residual_transformer_fixed_age_split_seed42,gender_worst_group_mae,0.3849,0.3264,0.4682
...,...,...,...,...,...
65,residual_transformer_fixed_age_split_seed8192,r2,0.0823,-0.2793,0.3357
66,residual_transformer_fixed_age_split_seed8192,gender_gap,0.0530,0.0010,0.1447
67,residual_transformer_fixed_age_split_seed8192,gender_worst_group_mae,0.3504,0.2859,0.4424
68,residual_transformer_fixed_age_split_seed8192,age_gap,0.1397,0.0391,0.2301


,subject_id,samples,true_mean,mean_mae,sd_mae,min_mae,max_mae
8,group02_subject_20,2459,2.7932,0.4545,0.0324,0.4217,0.5188
3,group01_subject_13,1490,2.5522,0.4482,0.0482,0.3673,0.5275
1,group01_subject_07,1614,2.6640,0.4412,0.0336,0.3758,0.4866
6,group02_subject_12,1967,2.6833,0.4306,0.0578,0.3628,0.5564
0,group01_subject_02,1610,2.8135,0.4014,0.0491,0.3406,0.5085
11,group03_subject_14,2478,3.1217,0.3958,0.0566,0.3125,0.4627
5,group02_subject_07,1050,3.0031,0.2878,0.0274,0.2347,0.3225
7,group02_subject_13,1408,2.8203,0.2737,0.0333,0.2146,0.3228
4,group02_subject_05,2430,3.1334,0.2712,0.0209,0.2471,0.3144
10,group03_subject_12,2488,3.2776,0.2628,0.0387,0.2235,0.3471


In [59]:
# Paired robustness comparison against the completed visual baseline.
# Positive MAE gain means residual Transformer reduced error relative to visual-only.
ROBUSTNESS_SUBSETS = {
    "all_test_windows": None,
    "at_least_2_missing_images": "at_least_2_missing_images",
    "at_least_4_missing_images": "at_least_4_missing_images",
}
PAIRED_BOOTSTRAP_RUNS = 1000
VISUAL_RESULTS_DIR = "results/Temporal Visual Baseline Stability"


def paired_subject_bootstrap_mae_gain(
    baseline_frame,
    candidate_frame,
    subset_col=None,
    n_boot=PAIRED_BOOTSTRAP_RUNS,
    seed=SEED,
):
    keys = ["subject_experiment_id", "time_sec"]
    metadata_cols = ["subject_id"]
    if subset_col is not None:
        metadata_cols.append(subset_col)

    paired = baseline_frame[keys + metadata_cols + ["true", "pred"]].merge(
        candidate_frame[keys + ["pred"]],
        on=keys,
        how="inner",
        suffixes=("_visual", "_fusion"),
    )
    if subset_col is not None:
        paired = paired[paired[subset_col].fillna(False).astype(bool)]

    paired["mae_gain"] = (
        np.abs(paired["pred_visual"] - paired["true"])
        - np.abs(paired["pred_fusion"] - paired["true"])
    )
    subjects = paired["subject_id"].dropna().unique()
    if len(subjects) == 0:
        return {
            "n_samples": 0,
            "n_subjects": 0,
            "mae_gain": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    grouped = {
        subject: group
        for subject, group in paired.groupby("subject_id", sort=False)
    }
    rng = np.random.default_rng(seed)
    gains = []
    for _ in range(n_boot):
        sampled_subjects = rng.choice(subjects, size=len(subjects), replace=True)
        sampled = pd.concat(
            [grouped[subject] for subject in sampled_subjects],
            ignore_index=True,
        )
        gains.append(float(sampled["mae_gain"].mean()))

    return {
        "n_samples": int(len(paired)),
        "n_subjects": int(len(subjects)),
        "mae_gain": float(paired["mae_gain"].mean()),
        "ci_low": float(np.quantile(gains, 0.025)),
        "ci_high": float(np.quantile(gains, 0.975)),
    }


paired_robustness_rows = []
paired_bootstrap_rows = []

for run_seed in RUN_SEEDS:
    visual_name = f"visual_baseline_fixed_age_split_seed{run_seed}"
    fusion_name = f"residual_transformer_fixed_age_split_seed{run_seed}"
    visual_path = os.path.join(VISUAL_RESULTS_DIR, f"{visual_name}_test_predictions.csv")

    if not os.path.exists(visual_path):
        print(f"Skipping paired robustness comparison; missing {visual_path}")
        continue

    visual_frame = pd.read_csv(visual_path)
    visual_frame = ev.add_robustness_metadata(
        visual_frame,
        test_df,
        visual_missing_col="visual_missing_flags",
    )
    fusion_frame = test_predictions[fusion_name]

    comparison = ev.paired_robustness_comparison(
        {"visual": visual_frame, "residual_transformer": fusion_frame},
        baseline_model="visual",
        candidate_model="residual_transformer",
        subsets=ROBUSTNESS_SUBSETS,
    )
    comparison.insert(0, "run_seed", run_seed)
    paired_robustness_rows.append(comparison)

    for subset_name, subset_col in ROBUSTNESS_SUBSETS.items():
        result = paired_subject_bootstrap_mae_gain(
            visual_frame,
            fusion_frame,
            subset_col=subset_col,
            n_boot=PAIRED_BOOTSTRAP_RUNS,
            seed=SEED + run_seed,
        )
        paired_bootstrap_rows.append({
            "run_seed": run_seed,
            "subset": subset_name,
            **result,
        })

paired_robustness_summary = pd.concat(
    paired_robustness_rows,
    ignore_index=True,
) if paired_robustness_rows else pd.DataFrame()
paired_robustness_bootstrap = pd.DataFrame(paired_bootstrap_rows)

display(paired_robustness_summary.round(4))
display(paired_robustness_bootstrap.round(4))


/gpfs/home5/cfragkiadakis/thesis/src/evaluation.py:170: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = paired[paired[subset_col].fillna(False).astype(bool)]
/gpfs/home5/cfragkiadakis/thesis/src/evaluation.py:170: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = paired[paired[subset_col].fillna(False).astype(bool)]
/scratch-local/cfragkiadakis.23868268/ipykernel_4009441/1739976948.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=

,run_seed,baseline_model,candidate_model,subset,n_samples,baseline_mae,candidate_mae,candidate_mae_gain,candidate_better_rate
0,42,visual,residual_transformer,all_test_windows,5122,0.2541,0.3373,-0.0832,0.3565
1,42,visual,residual_transformer,at_least_2_missing_images,509,0.2975,0.3706,-0.0732,0.3458
2,42,visual,residual_transformer,at_least_4_missing_images,39,0.4377,0.3138,0.1239,0.7436
3,100,visual,residual_transformer,all_test_windows,5122,0.2494,0.3728,-0.1234,0.3469
4,100,visual,residual_transformer,at_least_2_missing_images,509,0.2662,0.3911,-0.1250,0.3576
5,100,visual,residual_transformer,at_least_4_missing_images,39,0.4304,0.3501,0.0803,0.6667
6,2000,visual,residual_transformer,all_test_windows,5122,0.2948,0.3756,-0.0809,0.3266
7,2000,visual,residual_transformer,at_least_2_missing_images,509,0.3342,0.4194,-0.0852,0.3163
8,2000,visual,residual_transformer,at_least_4_missing_images,39,0.4202,0.3457,0.0746,0.6667
9,2025,visual,residual_transformer,all_test_windows,5122,0.2745,0.3422,-0.0677,0.3733


,run_seed,subset,n_samples,n_subjects,mae_gain,ci_low,ci_high
0,42,all_test_windows,5122,3,-0.0832,-0.1426,0.0065
1,42,at_least_2_missing_images,509,3,-0.0732,-0.1519,0.0122
2,42,at_least_4_missing_images,39,3,0.1239,-0.0098,0.1953
3,100,all_test_windows,5122,3,-0.1234,-0.2146,-0.0082
4,100,at_least_2_missing_images,509,3,-0.1250,-0.2531,-0.0056
5,100,at_least_4_missing_images,39,3,0.0803,-0.0800,0.2719
6,2000,all_test_windows,5122,3,-0.0809,-0.1285,-0.0084
7,2000,at_least_2_missing_images,509,3,-0.0852,-0.1530,-0.0176
8,2000,at_least_4_missing_images,39,3,0.0746,0.0158,0.1448
9,2025,all_test_windows,5122,3,-0.0677,-0.1253,-0.0053


In [60]:
# Synthetic visual-missingness stress test.
# Natural missing images remain missing. Additional available images are hidden
# deterministically until each eligible 10-second window reaches the requested
# total missing-image count.
import hashlib

SYNTHETIC_MASK_LEVELS = [4, 6, 8]
SYNTHETIC_MASK_SEED = 20260612
SYNTHETIC_BOOTSTRAP_RUNS = 1000


def make_synthetic_masked_sequences(
    sequence_df,
    visual_missing_col,
    target_missing,
    mask_seed=SYNTHETIC_MASK_SEED,
):
    masked_rows = []

    for _, row in sequence_df.iterrows():
        flags = np.asarray(row[visual_missing_col], dtype=np.float32).copy()
        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64).copy()
        natural_missing = int(flags.sum())

        # Do not unmask naturally missing images. Windows already above the
        # requested level are excluded from that exact-level condition.
        if natural_missing > target_missing:
            continue

        additional_missing = target_missing - natural_missing
        candidates = np.flatnonzero((flags == 0) & (feature_rows >= 0))
        if len(candidates) < additional_missing:
            continue

        key = (
            f"{row['subject_experiment_id']}|{int(row['time_sec'])}|"
            f"{target_missing}|{mask_seed}"
        ).encode("utf-8")
        stable_seed = int.from_bytes(hashlib.sha256(key).digest()[:8], "little")
        rng = np.random.default_rng(stable_seed)
        chosen = rng.choice(candidates, size=additional_missing, replace=False)

        flags[chosen] = 1.0
        feature_rows[chosen] = -1

        updated = row.copy()
        updated[visual_missing_col] = flags.tolist()
        updated["feature_rows"] = feature_rows.tolist()
        updated["natural_visual_missing_count"] = natural_missing
        updated["synthetic_visual_missing_count"] = int(flags.sum())
        updated["synthetic_mask_level"] = target_missing
        masked_rows.append(updated)

    return pd.DataFrame(masked_rows).reset_index(drop=True)


def synthetic_seed_summary_table(seed_metrics):
    rows = []
    for level, group in seed_metrics.groupby("mask_level", observed=True):
        for metric in ["mae", "rmse", "r2"]:
            values = group[metric].astype(float)
            rows.append({
                "mask_level": int(level),
                "metric": metric,
                "mean": float(values.mean()),
                "std": float(values.std()),
                "min": float(values.min()),
                "max": float(values.max()),
            })
    return pd.DataFrame(rows)


synthetic_masking_predictions = {}
synthetic_masking_records = []
synthetic_masking_ensemble_rows = []
synthetic_masking_bootstrap_rows = []
paired_synthetic_rows = []
paired_synthetic_bootstrap_rows = []

fusion_loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "shuffle": False,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
    "prefetch_factor": 4 if NUM_WORKERS > 0 else None,
}
fusion_loader_kwargs = {
    key: value for key, value in fusion_loader_kwargs.items() if value is not None
}

for mask_level in SYNTHETIC_MASK_LEVELS:
    masked_test_df = make_synthetic_masked_sequences(
        test_df,
        visual_missing_col="visual_missing_flags",
        target_missing=mask_level,
    )
    masked_dataset = MultimodalFusionDataset(masked_test_df, feature_store_path)
    masked_loader = DataLoader(masked_dataset, **fusion_loader_kwargs)
    level_frames = []

    print(
        f"Synthetic missing={mask_level}: {len(masked_test_df)} windows, "
        f"{masked_test_df['subject_id'].nunique()} subjects"
    )

    for run_seed in RUN_SEEDS:
        run_name = f"residual_transformer_fixed_age_split_seed{run_seed}"
        model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")
        model = make_model().to(device)
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))

        metrics, frame = evaluate_loader(model, masked_loader, masked_test_df)
        frame["synthetic_mask_level"] = mask_level
        frame["synthetic_mask_seed"] = SYNTHETIC_MASK_SEED
        prediction_name = f"{run_name}_synthetic_missing{mask_level}"
        synthetic_masking_predictions[prediction_name] = frame
        level_frames.append(frame)
        synthetic_masking_records.append({
            "run_name": run_name,
            "run_seed": run_seed,
            "mask_level": mask_level,
            "n_samples": len(frame),
            "n_subjects": frame["subject_id"].nunique(),
            **metrics,
        })
        ev.save_prediction_frame(
            frame,
            os.path.join(RESULTS_DIR, f"{prediction_name}_predictions.csv"),
        )

        visual_path = os.path.join(
            "results/Temporal Visual Baseline Stability",
            f"visual_baseline_fixed_age_split_seed{run_seed}_synthetic_missing{mask_level}_predictions.csv",
        )
        if os.path.exists(visual_path):
            visual_frame = pd.read_csv(visual_path)
            comparison = ev.paired_robustness_comparison(
                {"visual": visual_frame, "residual_transformer": frame},
                baseline_model="visual",
                candidate_model="residual_transformer",
                subsets={f"exactly_{mask_level}_missing_images": None},
            )
            comparison.insert(0, "run_seed", run_seed)
            comparison.insert(1, "mask_level", mask_level)
            paired_synthetic_rows.append(comparison)

            paired_bootstrap = paired_subject_bootstrap_mae_gain(
                visual_frame,
                frame,
                subset_col=None,
                n_boot=SYNTHETIC_BOOTSTRAP_RUNS,
                seed=SYNTHETIC_MASK_SEED + mask_level + run_seed,
            )
            paired_synthetic_bootstrap_rows.append({
                "run_seed": run_seed,
                "mask_level": mask_level,
                **paired_bootstrap,
            })

    ensemble = level_frames[0].copy()
    ensemble["pred"] = np.mean(
        [frame["pred"].to_numpy(dtype=float) for frame in level_frames],
        axis=0,
    )
    ensemble_metrics = ev.compute_prediction_metrics(ensemble)
    synthetic_masking_ensemble_rows.append({
        "mask_level": mask_level,
        "n_subjects": ensemble["subject_id"].nunique(),
        **ensemble_metrics,
    })
    ev.save_prediction_frame(
        ensemble,
        os.path.join(
            RESULTS_DIR,
            f"synthetic_missing{mask_level}_seed_ensemble_predictions.csv",
        ),
    )

    bootstrap_summary_level, _ = ev.cluster_bootstrap_ci(
        ensemble,
        cluster_col="subject_id",
        n_boot=SYNTHETIC_BOOTSTRAP_RUNS,
        seed=SYNTHETIC_MASK_SEED + mask_level,
    )
    bootstrap_summary_level.insert(0, "mask_level", mask_level)
    synthetic_masking_bootstrap_rows.append(bootstrap_summary_level.reset_index(names="metric"))

synthetic_masking_seed_metrics = pd.DataFrame(synthetic_masking_records)
synthetic_masking_seed_summary = synthetic_seed_summary_table(synthetic_masking_seed_metrics)
synthetic_masking_ensemble_metrics = pd.DataFrame(synthetic_masking_ensemble_rows)
synthetic_masking_ensemble_bootstrap = pd.concat(
    synthetic_masking_bootstrap_rows,
    ignore_index=True,
)
paired_synthetic_masking_summary = pd.concat(
    paired_synthetic_rows,
    ignore_index=True,
) if paired_synthetic_rows else pd.DataFrame()
paired_synthetic_masking_bootstrap = pd.DataFrame(paired_synthetic_bootstrap_rows)

display(synthetic_masking_seed_summary.round(4))
display(synthetic_masking_ensemble_metrics.round(4))
display(paired_synthetic_masking_summary.round(4))
display(paired_synthetic_masking_bootstrap.round(4))


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Synthetic missing=4: 22815 windows, 12 subjects


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/tra

Synthetic missing=6: 23057 windows, 12 subjects


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/tra

...

/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/tra

,mask_level,metric,mean,std,min,max
0,4,mae,0.3464,0.0163,0.3213,0.3682
1,4,rmse,0.4412,0.0195,0.4109,0.4690
2,4,r2,-0.0206,0.0899,-0.1514,0.1164
3,6,mae,0.3557,0.0266,0.3234,0.4094
4,6,rmse,0.4507,0.0295,0.4124,0.5083
5,6,r2,-0.0608,0.1406,-0.3443,0.1150
6,8,mae,0.3768,0.0558,0.3257,0.5104
7,8,rmse,0.4725,0.0582,0.4149,0.6063
8,8,r2,-0.1758,0.3065,-0.9102,0.1053


,mask_level,n_subjects,n_samples,mae,rmse,r2,tolerant_accuracy,tolerance,one_off_accuracy,binary_threshold,binary_accuracy,binary_f1,binary_roc_auc,binary_confusion_matrix,true_mean,pred_mean
0,4,12,22815,0.3304,0.4241,0.0585,0.3158,0.15,0.9975,3.0,0.5856,0.6010,0.6210,"[[6241, 4547], [4907, 7120]]",2.9402,3.0426
1,6,12,23057,0.3327,0.4258,0.0566,0.3091,0.15,0.9979,3.0,0.5878,0.6097,0.6172,"[[6131, 4763], [4741, 7422]]",2.9400,3.0411
2,8,12,23130,0.3331,0.4250,0.0615,0.2978,0.15,0.9990,3.0,0.5827,0.5903,0.6107,"[[6525, 4393], [5259, 6953]]",2.9400,3.0197


,run_seed,mask_level,baseline_model,candidate_model,subset,n_samples,baseline_mae,candidate_mae,candidate_mae_gain,candidate_better_rate
0,42,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2582,0.3684,-0.1102,0.3437
1,100,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2639,0.3616,-0.0977,0.3968
2,2000,4,visual,residual_transformer,exactly_4_missing_images,5103,0.3078,0.3657,-0.0579,0.3582
3,2025,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2794,0.3304,-0.0509,0.4076
4,2026,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2587,0.3316,-0.0728,0.3821
5,2027,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2721,0.3347,-0.0626,0.3635
6,2048,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2455,0.3621,-0.1166,0.3047
7,4096,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2557,0.3457,-0.0899,0.3498
8,7000,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2687,0.3887,-0.1200,0.3118
9,8192,4,visual,residual_transformer,exactly_4_missing_images,5103,0.2727,0.3108,-0.0381,0.4131


,run_seed,mask_level,n_samples,n_subjects,mae_gain,ci_low,ci_high
0,42,4,5103,3,-0.1102,-0.1808,-0.0175
1,100,4,5103,3,-0.0977,-0.1930,0.0057
2,2000,4,5103,3,-0.0579,-0.0897,-0.0040
3,2025,4,5103,3,-0.0509,-0.0930,-0.0076
4,2026,4,5103,3,-0.0728,-0.1347,-0.0062
5,2027,4,5103,3,-0.0626,-0.1209,-0.0062
6,2048,4,5103,3,-0.1166,-0.1741,-0.0609
7,4096,4,5103,3,-0.0899,-0.1681,-0.0114
8,7000,4,5103,3,-0.1200,-0.1961,-0.0482
9,8192,4,5103,3,-0.0381,-0.0823,0.0191


In [61]:
split_demographics.to_csv(os.path.join(RESULTS_DIR, "split_demographics.csv"), index=False)
seed_results.to_csv(os.path.join(RESULTS_DIR, "seed_results.csv"), index=False)
seed_summary.to_csv(os.path.join(RESULTS_DIR, "seed_summary.csv"), index=False)
reference_table.to_csv(os.path.join(RESULTS_DIR, "reference_baselines.csv"), index=False)
bootstrap_summary.to_csv(os.path.join(RESULTS_DIR, "subject_bootstrap_summary.csv"), index=False)
subject_seed_metrics.to_csv(os.path.join(RESULTS_DIR, "subject_seed_metrics.csv"), index=False)
subject_stability.to_csv(os.path.join(RESULTS_DIR, "subject_stability.csv"), index=False)
ev.save_prediction_frame(ensemble_frame, os.path.join(RESULTS_DIR, "seed_ensemble_test_predictions.csv"))
synthetic_masking_seed_metrics.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_seed_metrics.csv"), index=False)
synthetic_masking_seed_summary.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_seed_summary.csv"), index=False)
synthetic_masking_ensemble_metrics.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_ensemble_metrics.csv"), index=False)
synthetic_masking_ensemble_bootstrap.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_ensemble_bootstrap.csv"), index=False)

paired_synthetic_masking_summary.to_csv(os.path.join(RESULTS_DIR, "paired_visual_synthetic_masking_summary.csv"), index=False)
paired_synthetic_masking_bootstrap.to_csv(os.path.join(RESULTS_DIR, "paired_visual_synthetic_masking_bootstrap.csv"), index=False)
paired_robustness_summary.to_csv(os.path.join(RESULTS_DIR, "paired_visual_robustness_summary.csv"), index=False)
paired_robustness_bootstrap.to_csv(os.path.join(RESULTS_DIR, "paired_visual_robustness_bootstrap.csv"), index=False)
def json_safe(value):
    """Recursively convert pandas/NumPy objects and non-JSON dictionary keys."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value
    
with open(os.path.join(RESULTS_DIR, "run_manifest.json"), "w") as file:
    json.dump(
        json_safe({
            "experiment": "Residual multimodal Transformer fixed constrained split stability",
            "split_stratify": STRATIFY_COLUMN,
            "split_random_state": SPLIT_RANDOM_STATE,
            "split_constraint_penalty": split_penalty,
            "split_balance_score": split_balance_score,
            "run_seeds": RUN_SEEDS,
            "loss_type": LOSS_TYPE,
            "early_stopping_metric": EARLY_STOPPING_METRIC,
            "mean_baseline_metrics": mean_baseline_metrics,
            "ensemble_metrics": ensemble_metrics,
            "runs": run_records,
        }),
        file,
        indent=2,
        default=str,
    )
print(f"Saved residual Transformer stability results to: {RESULTS_DIR}")

Saved residual Transformer stability results to: results/Multimodal Fusion Residual Transformer Stability602020


### Reading the results

Use `seed_summary.csv` to judge whether the fixed split produces consistently
acceptable predictive performance. The mean, standard deviation, and worst seed
matter more than the single best seed.

The fixed split should only be used for the later fairness experiment if its
baseline consistently beats the train-mean constant predictor and its MAE/R2
remain acceptable across seeds.


### Repeated subject-split generalizability evaluation

The fixed-split multi-seed experiment measures sensitivity to model
initialization for one selection of participants. This additional evaluation
instead measures sensitivity to participant selection. Ten distinct,
demographically constrained 70--15--15 subject-level train-validation-test splits are
paired with ten training seeds. The exact same split-seed pairs are used by the
unregularized, gender-regularized, and age-regularized models.

Fairness regularization strengths are fixed to the values selected in the
original fixed-split validation experiments. They are not re-selected for each
new split. Consequently, the repeated-split test results evaluate whether the
previously selected interventions generalize to different held-out subjects.


In [24]:
# Rehydrate trained fixed-split residual Transformer runs without retraining.
# This requires the .pt checkpoint files saved by the original training loop.

import os
import torch

# If the notebook path has the old "Stability146" suffix but your outputs are elsewhere,
# this will try both common locations.
candidate_model_dirs = [
    MODEL_DIR,
    "models/Multimodal Fusion Residual Transformer Stability602020",
    "models/Multimodal Fusion Residual Transformer Stability602020",
]

candidate_results_dirs = [
    RESULTS_DIR,
    "results/Multimodal Fusion Residual Transformer Stability602020",
    "results/Multimodal Fusion Residual Transformer Stability602020",
]

MODEL_DIR = next((p for p in candidate_model_dirs if os.path.isdir(p)), MODEL_DIR)
RESULTS_DIR = next((p for p in candidate_results_dirs if os.path.isdir(p)), RESULTS_DIR)

run_records = []
test_predictions = {}
missing_checkpoints = []

for run_seed in RUN_SEEDS:
    run_name = f"residual_transformer_fixed_age_split_seed{run_seed}"
    model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")

    if not os.path.exists(model_path):
        missing_checkpoints.append(model_path)
        continue

    print(f"Loading {run_name} from {model_path}")

    model = make_model().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    test_metrics, test_frame = evaluate_loader(model, test_loader, test_df)
    test_predictions[run_name] = test_frame

    val_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_val_predictions.csv")
    test_prediction_path = os.path.join(RESULTS_DIR, f"{run_name}_test_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_prediction_path)
    ev.save_prediction_frame(test_frame, test_prediction_path)

    run_records.append({
        "run_name": run_name,
        "run_seed": run_seed,
        "best_epoch": None,
        "num_epochs_run": 0,
        "model_path": model_path,
        "val_prediction_path": val_prediction_path,
        "test_prediction_path": test_prediction_path,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "history": [],
    })

if missing_checkpoints:
    raise FileNotFoundError(
        "Cannot rehydrate the trained models because these checkpoint files are missing:\n"
        + "\n".join(missing_checkpoints[:5])
        + ("\n..." if len(missing_checkpoints) > 5 else "")
    )

print(f"Loaded {len(run_records)} trained runs without retraining.")

Loading residual_transformer_fixed_age_split_seed42 from models/Multimodal Fusion Residual Transformer Stability602020/residual_transformer_fixed_age_split_seed42.pt


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loading residual_transformer_fixed_age_split_seed100 from models/Multimodal Fusion Residual Transformer Stability602020/residual_transformer_fixed_age_split_seed100.pt


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loading residual_transformer_fixed_age_split_seed2000 from models/Multimodal Fusion Residual Transformer Stability602020/residual_transformer_fixed_age_split_seed2000.pt


...

/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loading residual_transformer_fixed_age_split_seed7000 from models/Multimodal Fusion Residual Transformer Stability602020/residual_transformer_fixed_age_split_seed7000.pt


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Loading residual_transformer_fixed_age_split_seed8192 from models/Multimodal Fusion Residual Transformer Stability602020/residual_transformer_fixed_age_split_seed8192.pt


Loaded 10 trained runs without retraining.


In [ ]:
# This section is intentionally separate from the fixed-split analysis.
REPEATED_SPLIT_COUNT = 10
REPEATED_SPLIT_RUN_SEEDS = [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]
REPEATED_SPLIT_SEARCH_TRIALS = 20_000
MIN_AGE_GROUP_SUBJECTS_VAL = 2
MIN_AGE_GROUP_SUBJECTS_TEST = 2
REPEATED_SPLIT_RESULTS_DIR = "results/Multimodal Fusion Residual Transformer Repeated Subject Splits"
REPEATED_SPLIT_MODEL_DIR = "models/Multimodal Fusion Residual Transformer Repeated Subject Splits"
REPEATED_SPLIT_MANIFEST_PATH = os.path.join(REPEATED_SPLIT_RESULTS_DIR, "split_manifest.csv")
os.makedirs(REPEATED_SPLIT_RESULTS_DIR, exist_ok=True)
os.makedirs(REPEATED_SPLIT_MODEL_DIR, exist_ok=True)


def generate_repeated_split_manifest(subject_frame):
    """Return the first ten valid distinct splits without using labels or predictions."""
    records = []
    signatures = set()
    all_age_groups = set(subject_frame["age_group"].astype(str))

    for offset in range(REPEATED_SPLIT_SEARCH_TRIALS):
        split_seed = SEED + offset
        try:
            train_subject_frame, temp_subject_frame = train_test_split(
                subject_frame,
                test_size=0.3,
                stratify=subject_frame["age_group"],
                random_state=split_seed,
            )
            validation_subject_frame, test_subject_frame = train_test_split(
                temp_subject_frame,
                test_size=0.5,
                stratify=temp_subject_frame["age_group"],
                random_state=split_seed,
            )
        except ValueError:
            continue

        valid = (
            validation_subject_frame["age_group"].astype(str).value_counts()
            .reindex(all_age_groups, fill_value=0).min()
            >= MIN_AGE_GROUP_SUBJECTS_VAL
            and test_subject_frame["age_group"].astype(str).value_counts()
            .reindex(all_age_groups, fill_value=0).min()
            >= MIN_AGE_GROUP_SUBJECTS_TEST
            and (validation_subject_frame["gender"].astype(str).str.lower() == "male").sum()
            >= MIN_MALE_VAL_SUBJECTS
            and (test_subject_frame["gender"].astype(str).str.lower() == "male").sum()
            >= MIN_MALE_TEST_SUBJECTS
        )
        if not valid:
            continue

        signature = (
            tuple(sorted(validation_subject_frame["subject_id"].astype(str))),
            tuple(sorted(test_subject_frame["subject_id"].astype(str))),
        )
        if signature in signatures:
            continue
        signatures.add(signature)

        split_id = len(signatures) - 1
        run_seed = REPEATED_SPLIT_RUN_SEEDS[split_id]
        for split_name, split_frame in [
            ("train", train_subject_frame),
            ("validation", validation_subject_frame),
            ("test", test_subject_frame),
        ]:
            for _, row in split_frame.iterrows():
                records.append({
                    "split_id": split_id,
                    "split_random_state": split_seed,
                    "run_seed": run_seed,
                    "split": split_name,
                    "subject_id": row["subject_id"],
                    "age_group": str(row["age_group"]),
                    "gender": str(row["gender"]),
                })
        if len(signatures) == REPEATED_SPLIT_COUNT:
            break

    if len(signatures) != REPEATED_SPLIT_COUNT:
        raise RuntimeError(
            f"Found only {len(signatures)} valid distinct repeated subject splits."
        )
    return pd.DataFrame(records)


def repeated_split_demographic_summary(manifest):
    return (
        manifest.groupby(["split_id", "split", "age_group", "gender"], observed=True)
        .size()
        .reset_index(name="subjects")
        .sort_values(["split_id", "split", "age_group", "gender"])
    )


def activate_repeated_split(split_id, manifest):
    """Replace the active notebook datasets/loaders with one saved subject split."""
    global train_df, val_df, test_df
    global train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader

    current = manifest[manifest["split_id"] == split_id]
    subject_sets = {
        split_name: set(current.loc[current["split"] == split_name, "subject_id"])
        for split_name in ["train", "validation", "test"]
    }
    if (
        subject_sets["train"] & subject_sets["validation"]
        or subject_sets["train"] & subject_sets["test"]
        or subject_sets["validation"] & subject_sets["test"]
    ):
        raise ValueError(f"Subject overlap detected in repeated split {split_id}.")

    frame_splits = {
        split_name: temporal_frame_dataset[
            temporal_frame_dataset["subject_id"].isin(subject_ids)
        ].copy()
        for split_name, subject_ids in subject_sets.items()
    }

    repeated_sensor_means = frame_splits["train"].loc[
        frame_splits["train"]["sensor_missing"] == 0, sensor_cols
    ].mean().fillna(0.0)
    repeated_sensor_stds = frame_splits["train"].loc[
        frame_splits["train"]["sensor_missing"] == 0, sensor_cols
    ].std().replace(0, np.nan).fillna(1.0)

    def scale(frame):
        frame = frame.copy()
        scaled = ((frame[sensor_cols] - repeated_sensor_means) / repeated_sensor_stds)
        scaled = scaled.astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0.0)
        scaled.loc[frame["sensor_missing"] == 1, :] = 0.0
        frame.loc[:, sensor_cols] = scaled.to_numpy(dtype=np.float32)
        return frame

    train_df = create_temporal_sequences(scale(frame_splits["train"]))
    val_df = create_temporal_sequences(scale(frame_splits["validation"]))
    test_df = create_temporal_sequences(scale(frame_splits["test"]))
    train_df, val_df, test_df, _ = add_attention_bin_weights(
        WEIGHT_ALPHA, train_df, val_df, test_df
    )

    train_dataset = MultimodalFusionDataset(train_df, feature_store_path)
    val_dataset = MultimodalFusionDataset(val_df, feature_store_path)
    test_dataset = MultimodalFusionDataset(test_df, feature_store_path)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=TRAIN_SHUFFLE,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0, prefetch_factor=4,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0, prefetch_factor=4,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0, prefetch_factor=4,
    )


def subject_balanced_group_metrics(frame, attribute):
    subject_errors = (
        frame.assign(abs_error=np.abs(frame["pred"].astype(float) - frame["true"].astype(float)))
        .groupby(["subject_id", attribute], observed=True)["abs_error"]
        .mean()
        .reset_index()
    )
    group_mae = subject_errors.groupby(attribute, observed=True)["abs_error"].mean()
    return float(group_mae.max()), float(group_mae.max() - group_mae.min())


repeated_split_manifest = generate_repeated_split_manifest(subject_df)
repeated_split_manifest.to_csv(REPEATED_SPLIT_MANIFEST_PATH, index=False)
repeated_split_demographics = repeated_split_demographic_summary(repeated_split_manifest)
repeated_split_demographics.to_csv(
    os.path.join(REPEATED_SPLIT_RESULTS_DIR, "split_demographics.csv"), index=False
)
display(repeated_split_demographics)
print(f"Saved shared repeated-split manifest to: {REPEATED_SPLIT_MANIFEST_PATH}")


,split_id,split,age_group,gender,subjects
0,0,test,"(13, 20]",female,4
1,0,test,"(20, 22]",female,3
2,0,test,"(22, 26]",female,1
3,0,test,"(22, 26]",male,2
4,0,test,"(26, 44]",female,1
...,...,...,...,...,...
182,9,validation,"(20, 22]",male,1
183,9,validation,"(22, 26]",female,1
184,9,validation,"(22, 26]",male,2
185,9,validation,"(26, 44]",female,1


Saved shared repeated-split manifest to: results/Multimodal Fusion Residual Transformer Repeated Subject Splits/split_manifest.csv


In [26]:
repeated_split_rows = []

for split_id in range(REPEATED_SPLIT_COUNT):
    split_config = repeated_split_manifest[
        repeated_split_manifest["split_id"] == split_id
    ].iloc[0]
    run_seed = int(split_config["run_seed"])
    split_random_state = int(split_config["split_random_state"])
    activate_repeated_split(split_id, repeated_split_manifest)
    set_global_seed(run_seed)

    run_name = f"residual_transformer_repeated_split{split_id:02d}_seed{run_seed}"
    model_path = os.path.join(REPEATED_SPLIT_MODEL_DIR, f"{run_name}.pt")
    model = make_model().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.3, patience=3
    )
    early_stopping = EarlyStopping(PATIENCE, model_path)
    run_train_loader = make_train_loader()
    history = []

    print(f"\n=== {run_name} | split random state {split_random_state} ===")
    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch_baseline(model, run_train_loader, optimizer, criterion)
        val_metrics, _ = evaluate_loader(model, val_loader, val_df)
        monitor = val_metrics[EARLY_STOPPING_METRIC]
        scheduler.step(monitor)
        history.append({
            "epoch": epoch + 1,
            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
        })
        if early_stopping.step(monitor, model, epoch):
            break

    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    test_metrics, test_predictions = evaluate_loader(model, test_loader, test_df)
    val_subject_age_worst, val_subject_age_gap = subject_balanced_group_metrics(
        val_predictions, "age_group"
    )
    val_subject_gender_worst, val_subject_gender_gap = subject_balanced_group_metrics(
        val_predictions, "gender"
    )
    test_subject_age_worst, test_subject_age_gap = subject_balanced_group_metrics(
        test_predictions, "age_group"
    )
    test_subject_gender_worst, test_subject_gender_gap = subject_balanced_group_metrics(
        test_predictions, "gender"
    )

    val_path = os.path.join(REPEATED_SPLIT_RESULTS_DIR, f"{run_name}_val_predictions.csv")
    test_path = os.path.join(REPEATED_SPLIT_RESULTS_DIR, f"{run_name}_test_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_path)
    ev.save_prediction_frame(test_predictions, test_path)

    row = {
        "model": "unregularized",
        "split_id": split_id,
        "split_random_state": split_random_state,
        "run_seed": run_seed,
        "best_epoch": early_stopping.best_epoch + 1,
        "num_epochs_run": len(history),
        "val_subject_age_worst_group_mae": val_subject_age_worst,
        "val_subject_age_gap": val_subject_age_gap,
        "val_subject_gender_worst_group_mae": val_subject_gender_worst,
        "val_subject_gender_gap": val_subject_gender_gap,
        "test_subject_age_worst_group_mae": test_subject_age_worst,
        "test_subject_age_gap": test_subject_age_gap,
        "test_subject_gender_worst_group_mae": test_subject_gender_worst,
        "test_subject_gender_gap": test_subject_gender_gap,
        "model_path": model_path,
        "val_prediction_path": val_path,
        "test_prediction_path": test_path,
    }
    row.update({
        f"val_{key}": value for key, value in val_metrics.items()
        if not isinstance(value, dict)
    })
    row.update({
        f"test_{key}": value for key, value in test_metrics.items()
        if not isinstance(value, dict)
    })
    repeated_split_rows.append(row)

repeated_split_results = pd.DataFrame(repeated_split_rows).sort_values("split_id")
repeated_summary_metrics = [
    "test_mae", "test_rmse", "test_r2",
    "test_age_worst_group_mae", "test_age_gap",
    "test_gender_worst_group_mae", "test_gender_gap",
    "test_subject_age_worst_group_mae", "test_subject_age_gap",
    "test_subject_gender_worst_group_mae", "test_subject_gender_gap",
]
repeated_split_summary = (
    repeated_split_results[repeated_summary_metrics]
    .agg(["mean", "std", "min", "median", "max"])
    .T.reset_index(names="metric")
)
repeated_split_results.to_csv(
    os.path.join(REPEATED_SPLIT_RESULTS_DIR, "repeated_split_results.csv"), index=False
)
repeated_split_summary.to_csv(
    os.path.join(REPEATED_SPLIT_RESULTS_DIR, "repeated_split_summary.csv"), index=False
)
display(repeated_split_results.round(4))
display(repeated_split_summary.round(4))
print(f"Saved repeated subject-split baseline results to: {REPEATED_SPLIT_RESULTS_DIR}")


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



=== residual_transformer_repeated_split00_seed42 | split random state 42 ===


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



=== residual_transformer_repeated_split01_seed100 | split random state 43 ===


/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


...

/home/cfragkiadakis/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



=== residual_transformer_repeated_split09_seed8192 | split random state 57 ===


,model,split_id,split_random_state,run_seed,best_epoch,num_epochs_run,val_subject_age_worst_group_mae,val_subject_age_gap,val_subject_gender_worst_group_mae,val_subject_gender_gap,test_subject_age_worst_group_mae,test_subject_age_gap,test_subject_gender_worst_group_mae,test_subject_gender_gap,model_path,val_prediction_path,test_prediction_path,val_mae,val_rmse,val_r2,val_true_mean,val_pred_mean,val_age_worst_group_mae,val_age_gap,val_gender_worst_group_mae,val_gender_gap,test_mae,test_rmse,test_r2,test_true_mean,test_pred_mean,test_age_worst_group_mae,test_age_gap,test_gender_worst_group_mae,test_gender_gap
0,unregularized,0,42,42,2,9,0.5053,0.2004,0.3643,0.0226,0.3251,0.0943,0.2911,0.0556,models/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split00_seed42.pt,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split00_seed42_val_predictions.csv,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split00_seed42_test_predictions.csv,0.3502,0.4502,-0.2000,2.8446,3.0154,0.5077,0.2035,0.3574,0.0212,0.2809,0.3624,0.1806,3.0553,3.0128,0.3253,0.0884,0.2914,0.0451
1,unregularized,1,43,100,4,11,0.3482,0.0620,0.3433,0.0364,0.3082,0.0713,0.3120,0.0842,models/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split01_seed100.pt,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split01_seed100_val_predictions.csv,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split01_seed100_test_predictions.csv,0.3284,0.4083,0.0315,2.8666,2.9806,0.3727,0.0865,0.3340,0.0121,0.2939,0.3800,0.1212,2.9983,3.0033,0.3144,0.0689,0.3115,0.0818
2,unregularized,2,47,2000,4,11,0.3656,0.0677,0.3572,0.0561,0.3811,0.1156,0.3441,0.0564,models/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split02_seed2000.pt,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split02_seed2000_val_predictions.csv,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split02_seed2000_test_predictions.csv,0.3362,0.4275,-0.0613,2.9912,3.0252,0.3577,0.0616,0.3571,0.0601,0.2991,0.3932,0.1328,2.9579,3.0197,0.3434,0.0813,0.3200,0.0330
3,unregularized,3,49,2025,6,13,0.3727,0.1002,0.3298,0.0249,0.3552,0.0253,0.3489,0.0190,models/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split03_seed2025.pt,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split03_seed2025_val_predictions.csv,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split03_seed2025_test_predictions.csv,0.3173,0.4003,0.0658,2.8865,2.9757,0.3615,0.0843,0.3247,0.0190,0.3318,0.4277,-0.1736,2.9261,2.8826,0.3497,0.0486,0.3374,0.0090
4,unregularized,4,50,2026,3,10,0.4492,0.1637,0.3870,0.0568,0.4087,0.1006,0.3547,0.0070,models/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split04_seed2026.pt,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split04_seed2026_val_predictions.csv,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split04_seed2026_test_predictions.csv,0.3327,0.4264,-0.0237,2.9999,2.9505,0.3936,0.1072,0.3397,0.0107,0.3511,0.4469,-0.0374,2.9401,3.0283,0.4055,0.1065,0.3635,0.0193
5,unregularized,5,53,2027,5,12,0.3336,0.0877,0.3036,0.0251,0.3392,0.0570,0.3256,0.0080,models/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split05_seed2027.pt,results/Multimodal Fusion Residual Transformer Repeated Subject Splits/residual_transformer_repeated_split05_seed2027_val_predictions.cs

,metric,mean,std,min,median,max
0,test_mae,0.3209,0.0242,0.2809,0.3230,0.3554
1,test_rmse,0.4124,0.0280,0.3624,0.4175,0.4487
2,test_r2,0.0124,0.1287,-0.2076,0.0456,0.1806
3,test_age_worst_group_mae,0.3604,0.0335,0.3144,0.3498,0.4145
4,test_age_gap,0.0847,0.0235,0.0486,0.0797,0.1304
5,test_gender_worst_group_mae,0.3306,0.0221,0.2914,0.3298,0.3635
6,test_gender_gap,0.0302,0.0218,0.0090,0.0245,0.0818
7,test_subject_age_worst_group_mae,0.3676,0.0387,0.3082,0.3681,0.4276
8,test_subject_age_gap,0.0926,0.0349,0.0253,0.0975,0.1500
9,test_subject_gender_worst_group_mae,0.3372,0.0243,0.2911,0.3389,0.3766


Saved repeated subject-split baseline results to: results/Multimodal Fusion Residual Transformer Repeated Subject Splits
